In [1]:
import os
from pathlib import Path

# set the root directory as the current working directory
os.chdir(Path.cwd().parent)
print(f"Current working directory: {os.getcwd()}")

Current working directory: /shared-docker/CyberSec-Reasoner


In [2]:
import torch
import random
import logging
import warnings

%load_ext autoreload
%autoreload 2

from datasets import DatasetDict
from src.utils.logger import setup_logging
from src.config_loader import load_config
from src.utils.utils import setup_env, log_gpu_info
from src.utils.seed import setup_seed
from src.utils.wandb import init_wandb, finish_wandb
from src.model_loader import load_tokenizer, load_qlora_base_model
from src.data_loader import load_dataset, apply_chat_tempalte
from src.dataset_stats import print_token_stats
from src.trainer import build_sft_config, build_trainer, save_adapter, merge_and_save
from src.utils.plots import plot_learning_curve

warnings.filterwarnings("ignore")
logger = logging.getLogger(__name__)

In [3]:
if torch.cuda.is_available():
    print(f"Number of available GPUs: {torch.cuda.device_count()}")
    print(f"GPU Name: {torch.cuda.get_device_name()}")
    print(f"Total GPU Memory: {torch.cuda.get_device_properties().total_memory / 1024**3:.2f} GB")
else:
    print("No GPU detected! Using CPU...")

Number of available GPUs: 1
GPU Name: 
Total GPU Memory: 191.69 GB


*****
# Training

In [4]:
## setup logging
setup_logging()

config_path = "./src/configs/sft_qwen3.5_4b_soft.yaml"
model_cfg, dataset_cfg, lora_config, training_cfg, wandb_cfg, paths_cfg, _ = load_config(config_path)

# setup environment
setup_env()

# log GPU info
log_gpu_info()

# setup seed
setup_seed(training_cfg)

# init wandb
wandb_run = init_wandb(training_cfg["report_to"], wandb_cfg)

2026-04-11 06:08:40 | INFO     | src.utils.logger | Logging is set up.
2026-04-11 06:08:40 | INFO     | src.config_loader | Config loaded from: ./src/configs/sft_qwen3.5_4b_soft.yaml
2026-04-11 06:08:40 | INFO     | src.config_loader | Config validation passed
2026-04-11 06:08:40 | INFO     | src.config_loader | Output directories created (if they did not exist)
2026-04-11 06:08:40 | INFO     | src.config_loader | Model configs: {'name': 'Qwen/Qwen3.5-4B', 'torch_dtype': 'bfloat16', 'load_in_8bit': True, 'llm_int8_threshold': 6.0, 'llm_int_skip_modules': 'none', 'llm_int8_enable_fp32_cpu_offload': True, 'attn_implementation': 'flash_attention_2', 'device_map': 'auto', 'trust_remote_code': True}
2026-04-11 06:08:40 | INFO     | src.config_loader | Dataset configs: {'path': './data/processed/cybersecurity_sft_dataset', 'max_length': 2048, 'column': 'messages', 'padding_side': 'right', 'truncation_side': 'left', 'packing': True}
2026-04-11 06:08:40 | INFO     | src.config_loader | LoRA co

wandb: WARNING The get_url method is deprecated and will be removed in a future release. Please use `run.url` instead.
2026-04-11 06:08:42 | INFO     | src.utils.wandb | WandB run 'qwen3.5-4b-lora-soft' initialized. Dashboard: https://wandb.ai/circuit/cybersec-reasoner/runs/e9rql44q


In [5]:
# tokenizer
tokenizer = load_tokenizer(model_cfg, dataset_cfg)

2026-04-11 06:08:55 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3.5-4B/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-04-11 06:08:55 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen3.5-4B/851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a/config.json "HTTP/1.1 200 OK"
2026-04-11 06:08:55 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3.5-4B/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-04-11 06:08:55 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen3.5-4B/851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a/tokenizer_config.json "HTTP/1.1 200 OK"
2026-04-11 06:08:55 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3.5-4B/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-04-11 06:08:55 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/model

In [6]:
# dataset
dataset = load_dataset(dataset_cfg)

# create smaller splits
sample_dataset = DatasetDict({
    "train": dataset["train"].select(range(1000)),
    "validation": dataset["validation"].select(range(200)),
    "test": dataset["test"].select(range(200)),
})

dataset = sample_dataset

# apply chat template
dataset = apply_chat_tempalte(dataset, tokenizer, "qwen")
print_token_stats(dataset, tokenizer)

2026-04-11 06:08:56 | INFO     | src.data_loader | Dataset loaded from data/processed/cybersecurity_sft_dataset with 96359 training, 5353 validation, 5354 test samples.
2026-04-11 06:08:56 | INFO     | src.data_loader | Applying chat template for model family: qwen


Dataset: DatasetDict({
    train: Dataset({
        features: ['messages'],
        num_rows: 96359
    })
    validation: Dataset({
        features: ['messages'],
        num_rows: 5353
    })
    test: Dataset({
        features: ['messages'],
        num_rows: 5354
    })
})


2026-04-11 06:08:57 | INFO     | src.data_loader | Chat template applied to dataset.
Processing train: 100% 1000/1000 [00:01<00:00, 692.03it/s]
2026-04-11 06:08:58 | INFO     | src.dataset_stats | [train] stats -> Min: 292 | Avg: 823.04 | Max: 2374
Processing validation: 100% 200/200 [00:00<00:00, 738.64it/s]
2026-04-11 06:08:58 | INFO     | src.dataset_stats | [validation] stats -> Min: 480 | Avg: 804.61 | Max: 2248
Processing test: 100% 200/200 [00:00<00:00, 701.43it/s]
2026-04-11 06:08:59 | INFO     | src.dataset_stats | [test] stats -> Min: 513 | Avg: 830.18 | Max: 2751


In [ ]:
# # create smaller splits
# sample_dataset = DatasetDict({
#     "train": dataset["train"].select(range(1000)),
#     "validation": dataset["validation"].select(range(200)),
#     "test": dataset["test"].select(range(200)),
# })

# dataset = sample_dataset
# print(dataset)
# print_token_stats(dataset, tokenizer)

In [10]:
# print a random sample from the training dataset after applying the chat template
print(dataset["train"][random.randint(0, (len(dataset["train"]) -1))]["text"])

<|im_start|>system
You are a cybersecurity expert. Analyze the given vulnerability context carefully and think step by step to understand the root cause, risk, and impact. Then provide a clear, concise, and accurate answer to the user's question based on your reasoning.<|im_end|>
<|im_start|>user
cwe_id:CWE-77
cwe_name:Improper Neutralization of Special Elements used in a Command ('Command Injection')
affected_line:Command Injection in lodash
partial_code:lodash 4.17.19
file_name:yarn.lock
status:True Positive
reason: The lodash version 4.17.19 is vulnerable to Command Injection via the template function.
remediation_action: Update the lodash to version 4.17.21 or higher.

How to fix this?<|im_end|>
<|im_start|>assistant
<think>
First, the user is asking "How to fix this?" based on the provided details. I need to remember my system prompt: I am to provide a short, pinpointed answer that directly addresses how to fix the issue, using only the exact steps without additional explanation.


In [11]:
# load model
log_gpu_info("Before loading model")
model = load_qlora_base_model(model_cfg)
log_gpu_info("After loading model")

2026-04-11 06:09:34 | INFO     | src.utils.utils | GPU 0: , 205.82 GB VRAM. Before loading model
2026-04-11 06:09:34 | INFO     | src.model_loader | Loading QLoRA model: Qwen/Qwen3.5-4B with bitsandbytes config: BitsAndBytesConfig {
  "_load_in_4bit": false,
  "_load_in_8bit": true,
  "bnb_4bit_compute_dtype": "float32",
  "bnb_4bit_quant_storage": "uint8",
  "bnb_4bit_quant_type": "fp4",
  "bnb_4bit_use_double_quant": false,
  "llm_int8_enable_fp32_cpu_offload": true,
  "llm_int8_has_fp16_weight": false,
  "llm_int8_skip_modules": null,
  "llm_int8_threshold": 6.0,
  "load_in_4bit": false,
  "load_in_8bit": true,
  "quant_method": "bitsandbytes"
}

2026-04-11 06:09:34 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3.5-4B/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-04-11 06:09:34 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen3.5-4B/851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a/config.json "HTTP/1

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/426 [00:00<?, ?it/s]

2026-04-11 06:09:41 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3.5-4B/resolve/main/generation_config.json "HTTP/1.1 404 Not Found"
2026-04-11 06:09:41 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3.5-4B/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-04-11 06:09:41 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen3.5-4B/851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a/config.json "HTTP/1.1 200 OK"
2026-04-11 06:09:41 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3.5-4B/resolve/main/custom_generate/generate.py "HTTP/1.1 404 Not Found"
2026-04-11 06:09:41 | INFO     | src.model_loader | QLoRA model Qwen/Qwen3.5-4B loaded with 4,205,751,296 parameters.
2026-04-11 06:09:41 | INFO     | src.model_loader | Model Loaded on device: cuda:0
2026-04-11 06:09:41 | INFO     | src.utils.utils | GPU 0: , 205.82 GB VRAM. After loading model


In [12]:
# init trainer
sft_config = build_sft_config(dataset_cfg, training_cfg, wandb_cfg["run_name"])

trainer = build_trainer(
    model, 
    tokenizer, 
    dataset,
    sft_config,
    lora_config
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.
2026-04-11 06:09:41 | INFO     | src.trainer | Create LoRA Configs...
2026-04-11 06:09:41 | INFO     | src.model_loader | LoRA config created with r=64, alpha=128, dropout=0.05, bias=none, task_type=CAUSAL_LM, use_rslora=True,
target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'], 
2026-04-11 06:09:41 | INFO     | src.trainer | Prepare the model for LoRA fine-tuning...
2026-04-11 06:09:42 | INFO     | src.trainer | Peft Model Info: None
2026-04-11 06:09:42 | INFO     | src.trainer | Innitializing the SFTTrainer...


trainable params: 84,934,656 || all params: 4,290,685,952 || trainable%: 1.9795


2026-04-11 06:09:42 | INFO     | src.trainer | SFTTrainer initialized successfully.
2026-04-11 06:09:42 | INFO     | src.trainer | Number of trainable parameters    : 84,934,656
2026-04-11 06:09:42 | INFO     | src.trainer | Number of non-trainable parameters: 4,205,751,296
2026-04-11 06:09:42 | INFO     | src.trainer | Effective batch size (per_device * grad_accum): 64
2026-04-11 06:09:42 | INFO     | src.trainer | Steps per epoch                   : 16
2026-04-11 06:09:42 | INFO     | src.trainer | Total training steps              : 48


In [13]:
# start training
log_gpu_info("Before training")
logger.info("Starting training...")
result = trainer.train()
log_gpu_info("After training")
logger.info(f"Training completed. Training result: {result}")

2026-04-11 06:09:42 | INFO     | src.utils.utils | GPU 0: , 205.82 GB VRAM. Before training
2026-04-11 06:09:42 | INFO     | __main__ | Starting training...
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046, 'pad_token_id': 248044}.


Step,Training Loss,Validation Loss
10,1.579856,0.848165
20,0.655453,0.792778
21,0.655453,0.793178


2026-04-11 07:08:06 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3.5-4B/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-04-11 07:08:06 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen3.5-4B/851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a/config.json "HTTP/1.1 200 OK"
2026-04-11 07:08:06 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3.5-4B/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-04-11 07:08:06 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen3.5-4B/851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a/config.json "HTTP/1.1 200 OK"
2026-04-11 07:08:08 | INFO     | src.utils.utils | GPU 0: , 205.82 GB VRAM. After training
2026-04-11 07:08:08 | INFO     | __main__ | Training completed. Training result: TrainOutput(global_step=21, training_loss=1.0940706304141454, metrics={'train_runtime': 3505.08, 'train_samples_per_second

In [14]:
result.metrics

{'train_runtime': 3505.08,
 'train_samples_per_second': 0.365,
 'train_steps_per_second': 0.006,
 'total_flos': 5.369062965096038e+16,
 'train_loss': 1.0940706304141454}

In [15]:
metrics = result.metrics
trainer.log_metrics("train", metrics)
trainer.save_metrics("train", metrics)

plot_learning_curve(trainer, "Qwen3.5 9B ISFT Learning Curve", paths_cfg["plots_dir"])

2026-04-11 07:08:17 | INFO     | src.utils.plots | Qwen3.5 9B ISFT Learning Curve saved at: ./plots/Qwen3.5 9B ISFT Learning Curve.png


***** train metrics *****
  total_flos               = 50003295GF
  train_loss               =     1.0941
  train_runtime            = 0:58:25.07
  train_samples_per_second =      0.365
  train_steps_per_second   =      0.006


In [16]:
# save
save_adapter(trainer, tokenizer, paths_cfg)
merge_and_save(model_cfg, paths_cfg)

2026-04-11 07:08:27 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3.5-4B/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-04-11 07:08:27 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen3.5-4B/851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a/config.json "HTTP/1.1 200 OK"
2026-04-11 07:08:27 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3.5-4B/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-04-11 07:08:27 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen3.5-4B/851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a/config.json "HTTP/1.1 200 OK"
2026-04-11 07:08:27 | INFO     | src.trainer | LoRA adapter saved to: ./models/adapter/qwen3.5-4b-lora-soft/
2026-04-11 07:08:27 | INFO     | src.trainer | Tokenizer saved to: ./models/qwen3.5-4b-lora-soft/
2026-04-11 07:08:27 | INFO     | src.trainer |    README.md 0.01 MB
2026-04-11 07:08:27

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/426 [00:00<?, ?it/s]

2026-04-11 07:08:31 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3.5-4B/resolve/main/generation_config.json "HTTP/1.1 404 Not Found"
2026-04-11 07:08:31 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3.5-4B/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-04-11 07:08:31 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen3.5-4B/851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a/config.json "HTTP/1.1 200 OK"
2026-04-11 07:08:31 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen3.5-4B/resolve/main/custom_generate/generate.py "HTTP/1.1 404 Not Found"


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

2026-04-11 07:08:41 | INFO     | src.trainer | Merged model saved to: ./models/qwen3.5-4b-lora-soft/


Qwen3_5ForCausalLM(
  (model): Qwen3_5TextModel(
    (embed_tokens): Embedding(248320, 2560)
    (layers): ModuleList(
      (0-2): 3 x Qwen3_5DecoderLayer(
        (linear_attn): Qwen3_5GatedDeltaNet(
          (act): SiLUActivation()
          (conv1d): Conv1d(8192, 8192, kernel_size=(4,), stride=(1,), padding=(3,), groups=8192, bias=False)
          (norm): Qwen3_5RMSNormGated()
          (out_proj): Linear(in_features=4096, out_features=2560, bias=False)
          (in_proj_qkv): Linear(in_features=2560, out_features=8192, bias=False)
          (in_proj_z): Linear(in_features=2560, out_features=4096, bias=False)
          (in_proj_b): Linear(in_features=2560, out_features=32, bias=False)
          (in_proj_a): Linear(in_features=2560, out_features=32, bias=False)
        )
        (mlp): Qwen3_5MLP(
          (gate_proj): Linear(in_features=2560, out_features=9216, bias=False)
          (up_proj): Linear(in_features=2560, out_features=9216, bias=False)
          (down_proj): Linear(

In [17]:
finish_wandb(
    wandb_run
)
logger.info("Wandb run finished.")

eval/entropy,█▁▁
eval/loss,█▁▁
eval/mean_token_accuracy,▁██
eval/num_tokens,▁██
eval/runtime,█▃▁
eval/samples_per_second,▁▆█
eval/steps_per_second,▁▆█
train/entropy,█▁
train/epoch,▁▁▇▇██
train/global_step,▁▁▇▇██
+5,...


2026-04-11 07:08:42 | INFO     | src.utils.wandb | WandB run finished and synced.
2026-04-11 07:08:42 | INFO     | __main__ | Wandb run finished.
